<a href="https://colab.research.google.com/github/klausreitz/cohid/blob/main/Corre%C3%A7%C3%A3o_monet%C3%A1ria_IPCA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Função para correção monetária pelo IPCA de série de valores

import requests
import xml.etree.ElementTree as ET
import pandas as pd

def corrigir_ipca(
    dados,
    data_referencia=None
):
    """
    Corrige valores monetários pelo IPCA (SIDRA – tabela 1737),
    aplicando o IPCA do mês seguinte ao evento.

    Parâmetros
    ----------
    dados : list of tuples
        Lista no formato [(data_str, valor), ...]
        Ex: [("28/09/2021", 350.00)]

    data_referencia : str ou None
        Data final da correção no formato 'MM/YYYY' ou 'YYYY-MM'.
        Se None, usa o último IPCA disponível.

    Retorno
    -------
    DataFrame pandas com valores originais e corrigidos.
    """

    # ------------------------------------------------------
    # 1) Baixar IPCA mensal oficial (SIDRA – tabela 1737)
    # ------------------------------------------------------

    URL_IPCA = "https://apisidra.ibge.gov.br/values/t/1737/n1/1/v/63/p/all?formato=xml"

    response = requests.get(URL_IPCA, timeout=30)
    response.raise_for_status()

    root = ET.fromstring(response.content)

    registros = []

    for item in root.findall(
        ".//{http://schemas.datacontract.org/2004/07/IBGE.BTE.Tabela}ValorDescritoPorSuasDimensoes"
    ):
        periodo = item.find("{*}D3C")
        valor = item.find("{*}V")

        try:
            ipca = float(valor.text.replace(",", "."))
        except:
            continue

        registros.append({
            "periodo": periodo.text,
            "ipca_mensal": ipca
        })

    ipca_df = pd.DataFrame(registros)
    ipca_df["periodo"] = pd.to_datetime(ipca_df["periodo"], format="%Y%m")
    ipca_df = ipca_df.sort_values("periodo").reset_index(drop=True)

    # ------------------------------------------------------
    # 2) Criar índice IPCA acumulado (base = 100)
    # ------------------------------------------------------

    ipca_df["fator_mensal"] = 1 + ipca_df["ipca_mensal"] / 100
    ipca_df["indice_ipca"] = 100 * ipca_df["fator_mensal"].cumprod()

    # ------------------------------------------------------
    # 3) Preparar dados do usuário
    # ------------------------------------------------------

    df = pd.DataFrame(dados, columns=["data_original", "valor_original"])
    df["data_original"] = pd.to_datetime(df["data_original"], format="%d/%m/%Y")

    # IPCA aplicável = mês seguinte ao evento
    df["competencia_ipca"] = (
        df["data_original"]
        .dt.to_period("M")
        .dt.to_timestamp()
        + pd.offsets.MonthBegin(1)
    )

    # ------------------------------------------------------
    # 4) Definir data de referência
    # ------------------------------------------------------

    if data_referencia:
        data_ref = pd.to_datetime("01/" + data_referencia, dayfirst=True)
        ipca_df_ref = ipca_df[ipca_df["periodo"] <= data_ref]
    else:
        ipca_df_ref = ipca_df.copy()

    periodo_final = ipca_df_ref["periodo"].max()
    indice_final = ipca_df_ref.loc[
        ipca_df_ref["periodo"] == periodo_final,
        "indice_ipca"
    ].iloc[0]

    # ------------------------------------------------------
    # 5) Correção monetária
    # ------------------------------------------------------

    df = df.merge(
        ipca_df_ref[["periodo", "indice_ipca"]],
        left_on="competencia_ipca",
        right_on="periodo",
        how="left"
    )

    df["fator_correcao"] = indice_final / df["indice_ipca"]
    df["valor_corrigido"] = df["valor_original"] * df["fator_correcao"]

    # ------------------------------------------------------
    # 6) Formatação final
    # ------------------------------------------------------

    resultado = df[[
        "data_original",
        "valor_original",
        "valor_corrigido"
    ]].copy()

    resultado["valor_original"] = resultado["valor_original"].map(
        lambda x: f"R$ {x:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")
    )

    resultado["valor_corrigido"] = resultado["valor_corrigido"].map(
        lambda x: f"R$ {x:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")
    )

    print(
        f"Índice IPCA de referência: "
        f"{periodo_final.strftime('%m/%Y')}"
    )

    return resultado
